In [1]:
from pathlib import Path
import re
import json
import hashlib
import warnings
from typing import Dict, List, Tuple, Optional, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except Exception:
    HAS_GEOPANDAS = False

try:
    import pyarrow.parquet as pq
    HAS_PYARROW = True
except Exception:
    HAS_PYARROW = False

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 240)

print("Imports loaded.")
print("geopandas available:", HAS_GEOPANDAS)
print("pyarrow available:", HAS_PYARROW)

Imports loaded.
geopandas available: True
pyarrow available: True


In [ ]:
# CONFIG

PROJECT_ROOT = Path("PROJECT_ROOT")
DATA_DIR = PROJECT_ROOT / "data"
ROOT = PROJECT_ROOT / "simulated_trajectory"
TRACKING_FILE = DATA_DIR / "tiger_leopard_env_tg.csv"

# Focal pairs requested for sensitivity comparison.
PAIR_IDS = [
    (131343, 37821),  # FT1-ML2
    (131343, 37823),  # FT1-ML4
    (229032, 37821),  # MT3-ML2
]

# Display labels used in manuscript figures/tables.
IDCOLLAR_MAPPING = {
    "131343": "FT1",
    "229011": "FT2",
    "229041": "FT3",
    "229012": "MT1",
    "229022": "MT2",
    "229032": "MT3",
    "31899": "FL",
    "31898": "ML1",
    "37821": "ML2",
    "37822": "ML3",
    "37823": "ML4",
}
PAIR_LABELS = {tuple(p): f"{IDCOLLAR_MAPPING.get(str(p[0]), p[0])}-{IDCOLLAR_MAPPING.get(str(p[1]), p[1])}" for p in PAIR_IDS}

# Compare the same number of Monte Carlo runs for every window length.
# If 1000 h has 1000 MC files, this notebook uses MC0001--MC0500 by default.
MAX_MC_COMPARE = 500
MC_INDEX = pd.Index(range(1, MAX_MC_COMPARE + 1), name="MC_round")

# Window runs and possible folder/batch names.
RUNS = {
    500: {
        "candidate_dirs": [
            "SENS_V2_WIN500_MC0001_0500",
            "SENS_WIN500_MC0001_0500",
        ],
        "batch_label_candidates": ["MC0001_0500", "MC0001_500"],
    },
    1000: {
        "candidate_dirs": [
            "CRW_slice_RT_WIN1000_MC0001_1000",
            "SENS_V2_WIN1000_MC0001_0500",
            "SENS_WIN1000_MC0001_0500",
        ],
        "batch_label_candidates": ["MC0001_0500", "MC0001_1000", "MC0001_500"],
    },
    2000: {
        "candidate_dirs": [
            "SENS_V2_WIN2000_MC0001_0500",
            "SENS_WIN2000_MC0001_0500",
        ],
        "batch_label_candidates": ["MC0001_0500", "MC0001_500"],
    },
}

# Time bins in the interaction outputs.
BIN_LABELS = ["0-1h", "1h-1d", "1-2d", "2-4d", "4-8d", "8-16d", "16-21d"]
CONCURRENT_BIN = "0-1h"
DELAYED_BINS = ["1h-1d", "1-2d", "2-4d", "4-8d", "8-16d", "16-21d"]

BIN_DISPLAY = {
    "0-1h": "0–1 h",
    "1h-1d": "1 h–1 d",
    "1-2d": "1–2 d",
    "2-4d": "2–4 d",
    "4-8d": "4–8 d",
    "8-16d": "8–16 d",
    "16-21d": "16–21 d",
}

# Duration 
DEFAULT_NUMERIC_DURATION_UNIT = "minutes"   # options: "minutes", "hours", "seconds"

# For R/M area calculations. If proj_lon/proj_lat are used, they are treated as meters.
UTM_EPSG = 32647
X_COL_CANDIDATES = ["proj_lon", "x", "X", "longitude", "lon", "Long", "LONG"]
Y_COL_CANDIDATES = ["proj_lat", "y", "Y", "latitude", "lat", "Lat", "LAT"]
TIME_COL_CANDIDATES = ["Time_LMT", "time", "datetime", "timestamp", "date_time"]
ID_COL = "idcollar"

# Optional manual common-coverage override for a pair label.
# Example:
# COMMON_COVERAGE_OVERRIDES = {"FT1-ML4": {"start": "2019-10-01 00:00:00", "end": "2020-03-01 00:00:00"}}
COMMON_COVERAGE_OVERRIDES = {}

OUT_DIR = ROOT / "sensitivity_common_coverage_sim_medians_observed_RM_outputs"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
for p in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("TRACKING_FILE:", TRACKING_FILE)
print("OUT_DIR:", OUT_DIR)
print("MAX_MC_COMPARE:", MAX_MC_COMPARE)

In [ ]:
# Resolve run folders and helper path utilities

def resolve_run_dir(root: Path, candidate_dirs: List[str]) -> Path:
    for name in candidate_dirs:
        p = root / name
        if p.exists():
            return p
    raise FileNotFoundError(
        "None of these candidate run folders exist:\n" +
        "\n".join(str(root / x) for x in candidate_dirs)
    )

def pair_id_string(id_pair: Tuple[int, int]) -> str:
    return f"{id_pair[0]}_{id_pair[1]}"

def pair_label(id_pair: Tuple[int, int]) -> str:
    return PAIR_LABELS.get(tuple(id_pair), f"{id_pair[0]}_{id_pair[1]}")

def pair_folder(run_dir: Path, id_pair: Tuple[int, int]) -> Path:
    return run_dir / f"by_slice_{id_pair[0]}_{id_pair[1]}_reverse_time"

def parse_slice_from_name(name) -> Optional[int]:
    m = re.search(r"slice0*(\d+)", str(name))
    return int(m.group(1)) if m else None

def parse_mc_from_name(name) -> Optional[int]:
    m = re.search(r"MC0*(\d+)", str(name))
    return int(m.group(1)) if m else None

def canonical_slice_label(slice_idx: int) -> str:
    return f"slice{int(slice_idx):03d}"

def list_slice_dirs(pdir: Path) -> List[Tuple[int, Path]]:
    if not pdir.exists():
        return []
    out = []
    for d in sorted(pdir.glob("slice*")):
        if d.is_dir():
            s = parse_slice_from_name(d.name)
            if s is not None:
                out.append((s, d))
    return out

def resolve_batch_label_for_pair(pdir: Path, batch_candidates: List[str]) -> str:
    """Find the first candidate batch folder present under any slice; fallback to first candidate."""
    for batch in batch_candidates:
        for _, sdir in list_slice_dirs(pdir):
            if (sdir / batch / "interactions").exists():
                return batch
    # fallback: inspect any MC folder under the first slice
    for _, sdir in list_slice_dirs(pdir):
        for bdir in sorted(sdir.glob("MC*")):
            if (bdir / "interactions").exists():
                return bdir.name
    return batch_candidates[0]

for wh, cfg in RUNS.items():
    cfg["run_dir"] = resolve_run_dir(ROOT, cfg["candidate_dirs"])
    print(f"{wh} h run:", cfg["run_dir"])

print("Path helpers ready.")

In [4]:
# Filename parsing, parquet readers, and time helpers

def parse_interaction_filename(fp: Path) -> Optional[dict]:
    """Parse one ORTEGA interaction output filename."""
    stem = Path(fp).stem
    sl = parse_slice_from_name(stem)
    mc = parse_mc_from_name(stem)

    if "_events_" in stem:
        kind = "events"
        after = stem.split("_events_", 1)[1]
    elif "_pairs_" in stem:
        kind = "pairs"
        after = stem.split("_pairs_", 1)[1]
    else:
        return None

    observed = after.endswith("_observed_time")
    if observed:
        time_bin = after.replace("_observed_time", "")
    else:
        time_bin = re.sub(r"_MC0*\d+$", "", after)

    return {
        "slice_idx": sl,
        "kind": kind,
        "time_bin": time_bin,
        "mc": mc,
        "observed": observed,
    }

def safe_read_parquet(fp: Path) -> pd.DataFrame:
    fp = Path(fp)
    if (not fp.exists()) or fp.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_parquet(fp)
    except Exception as e:
        print("Could not read:", fp, repr(e))
        return pd.DataFrame()

def standardize_time_bin_label(x) -> str:
    return str(x).replace("–", "-").replace("—", "-").strip()

def first_existing_column(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def to_datetime_series(s) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

EVENT_REFERENCE_TIME_CANDIDATES = [
    "event_ref_time", "reference_time", "ref_time", "event_time",
    "p1_start", "p1_t_start", "p1_time", "p1_Time_LMT",
    "p2_start", "p2_t_start", "p2_time", "p2_Time_LMT",
    "start", "start_time", "time", "Time_LMT",
]
EVENT_INTERVAL_COLUMN_CANDIDATES = [
    ("p1_start", "p1_end"),
    ("p1_t_start", "p1_t_end"),
    ("event_start", "event_end"),
    ("interaction_start", "interaction_end"),
    ("start_time", "end_time"),
    ("start", "end"),
    ("t_start", "t_end"),
    ("time_start", "time_end"),
    ("overlap_start", "overlap_end"),
]

def add_reference_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    c = first_existing_column(out, EVENT_REFERENCE_TIME_CANDIDATES)
    if c is not None:
        out["__ref_time"] = to_datetime_series(out[c])
    else:
        out["__ref_time"] = pd.NaT
    return out

def find_interval_cols(df: pd.DataFrame) -> Optional[Tuple[str, str]]:
    for a, b in EVENT_INTERVAL_COLUMN_CANDIDATES:
        if a in df.columns and b in df.columns:
            return a, b
    return None

def crop_by_common_coverage(df: pd.DataFrame, start, end, use_interval_overlap: bool = False) -> pd.DataFrame:
    if df is None or df.empty:
        return df.copy() if df is not None else pd.DataFrame()
    out = df.copy()
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)

    if use_interval_overlap:
        cols = find_interval_cols(out)
        if cols is not None:
            s = to_datetime_series(out[cols[0]])
            e = to_datetime_series(out[cols[1]])
            keep = (s < end) & (e > start)
            return out.loc[keep.fillna(False)].copy()

    out = add_reference_time(out)
    if out["__ref_time"].notna().any():
        keep = (out["__ref_time"] >= start) & (out["__ref_time"] < end)
        return out.loc[keep.fillna(False)].copy()
    return out.copy()

def duration_col_to_hours(values: pd.Series, col_name: str) -> pd.Series:
    vals = pd.to_numeric(values, errors="coerce").fillna(0.0)
    lc = str(col_name).lower()
    if lc.endswith("_h") or "hour" in lc:
        return vals
    if lc.endswith("_sec") or "second" in lc:
        return vals / 3600.0
    if lc.endswith("_min") or "minute" in lc:
        return vals / 60.0
    if lc == "duration":
        if DEFAULT_NUMERIC_DURATION_UNIT == "hours":
            return vals
        if DEFAULT_NUMERIC_DURATION_UNIT == "seconds":
            return vals / 3600.0
        return vals / 60.0
    # Conservative default.
    return vals / 60.0

def duration_column(df: pd.DataFrame) -> Optional[str]:
    candidates = ["duration_h", "duration_hours", "duration_min", "duration_minutes", "duration_sec", "duration_seconds", "duration"]
    return first_existing_column(df, candidates)

print("Parsing and time helpers ready.")

Parsing and time helpers ready.


In [5]:
# Load tracking data and compute common temporal coverage

def detect_time_col(df: pd.DataFrame) -> str:
    c = first_existing_column(df, TIME_COL_CANDIDATES)
    if c is None:
        raise ValueError(f"Could not find a time column. Candidates: {TIME_COL_CANDIDATES}")
    return c

def load_tracking_data() -> pd.DataFrame:
    if not TRACKING_FILE.exists():
        raise FileNotFoundError(f"TRACKING_FILE not found: {TRACKING_FILE}")
    df = pd.read_csv(TRACKING_FILE)
    tcol = detect_time_col(df)
    df["__time"] = pd.to_datetime(df[tcol], errors="coerce")
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")
    return df.dropna(subset=["__time", ID_COL]).copy()

tracking_df = load_tracking_data()
print("Tracking rows:", len(tracking_df))
print("Tracking time range:", tracking_df["__time"].min(), "to", tracking_df["__time"].max())

def shared_pair_tracking(df: pd.DataFrame, id_pair: Tuple[int, int]) -> pd.DataFrame:
    id1, id2 = id_pair
    sub = df[df[ID_COL].isin([id1, id2])].copy()
    if sub.empty:
        return sub
    per = sub.groupby(ID_COL)["__time"].agg(["min", "max"])
    if id1 not in per.index or id2 not in per.index:
        return pd.DataFrame(columns=sub.columns)
    shared_start = max(per.loc[id1, "min"], per.loc[id2, "min"])
    shared_end = min(per.loc[id1, "max"], per.loc[id2, "max"])
    return sub[(sub["__time"] >= shared_start) & (sub["__time"] <= shared_end)].copy()

def reverse_full_windows_for_pair(df: pd.DataFrame, id_pair: Tuple[int, int], window_h: int) -> List[Tuple[pd.Timestamp, pd.Timestamp]]:
    sub = shared_pair_tracking(df, id_pair)
    if sub.empty:
        return []
    start = sub["__time"].min()
    end = sub["__time"].max()
    window = pd.Timedelta(hours=window_h)
    rev = []
    cur_end = end
    while True:
        cur_start = cur_end - window
        if cur_start < start:
            break
        rev.append((cur_start, cur_end))
        cur_end = cur_start
    return list(reversed(rev))

def run_coverage_from_tracking(id_pair: Tuple[int, int], window_h: int) -> Tuple[pd.Timestamp, pd.Timestamp, int]:
    windows = reverse_full_windows_for_pair(tracking_df, id_pair, window_h)
    if not windows:
        return pd.NaT, pd.NaT, 0
    return windows[0][0], windows[-1][1], len(windows)

def common_coverage_for_pair(id_pair: Tuple[int, int]) -> dict:
    lab = pair_label(id_pair)
    if lab in COMMON_COVERAGE_OVERRIDES:
        o = COMMON_COVERAGE_OVERRIDES[lab]
        return {"pair": pair_id_string(id_pair), "pair_label": lab,
                "common_start": pd.to_datetime(o["start"]), "common_end": pd.to_datetime(o["end"]),
                "coverage_source": "manual_override"}
    rows = []
    for wh in RUNS:
        s, e, n = run_coverage_from_tracking(id_pair, wh)
        rows.append({"window_h": wh, "coverage_start": s, "coverage_end": e, "n_full_reverse_windows": n})
    d = pd.DataFrame(rows)
    common_start = d["coverage_start"].max()
    common_end = d["coverage_end"].min()
    return {"pair": pair_id_string(id_pair), "pair_label": lab,
            "common_start": common_start, "common_end": common_end,
            "coverage_source": "tracking_reverse_full_windows"}

coverage_rows = []
run_cov_rows = []
for p in PAIR_IDS:
    cov = common_coverage_for_pair(p)
    coverage_rows.append(cov)
    for wh in RUNS:
        s, e, n = run_coverage_from_tracking(p, wh)
        run_cov_rows.append({"pair": pair_id_string(p), "pair_label": pair_label(p), "window_h": wh,
                             "run_start": s, "run_end": e, "n_full_reverse_windows": n})

run_coverage = pd.DataFrame(run_cov_rows)
common_coverage = pd.DataFrame(coverage_rows)

run_coverage.to_csv(TABLE_DIR / "run_coverage_by_pair_window.csv", index=False)
common_coverage.to_csv(TABLE_DIR / "common_coverage_by_pair.csv", index=False)

print("Run coverage by pair/window:")
display(run_coverage)
print("Common coverage by pair:")
display(common_coverage)

Tracking rows: 91799
Tracking time range: 2017-12-20 07:00:00 to 2021-01-06 15:42:00
Run coverage by pair/window:


,pair,pair_label,window_h,run_start,run_end,n_full_reverse_windows
0,131343_37821,FT1-ML2,500,2019-10-13 11:00:00,2020-03-28 03:00:00,8
1,131343_37821,FT1-ML2,1000,2019-10-13 11:00:00,2020-03-28 03:00:00,4
2,131343_37821,FT1-ML2,2000,2019-10-13 11:00:00,2020-03-28 03:00:00,2
3,131343_37823,FT1-ML4,500,2019-11-07 07:00:00,2020-11-16 07:00:00,18
4,131343_37823,FT1-ML4,1000,2019-11-07 07:00:00,2020-11-16 07:00:00,9
5,131343_37823,FT1-ML4,2000,2019-12-18 23:00:00,2020-11-16 07:00:00,4
6,229032_37821,MT3-ML2,500,2019-10-13 11:00:00,2020-03-28 03:00:00,8
7,229032_37821,MT3-ML2,1000,2019-10-13 11:00:00,2020-03-28 03:00:00,4
8,229032_37821,MT3-ML2,2000,2019-10-13 11:00:00,2020-03-28 03:00:00,2


Common coverage by pair:


,pair,pair_label,common_start,common_end,coverage_source
0,131343_37821,FT1-ML2,2019-10-13 11:00:00,2020-03-28 03:00:00,tracking_reverse_full_windows
1,131343_37823,FT1-ML4,2019-12-18 23:00:00,2020-11-16 07:00:00,tracking_reverse_full_windows
2,229032_37821,MT3-ML2,2019-10-13 11:00:00,2020-03-28 03:00:00,tracking_reverse_full_windows


In [ ]:
# Load events and pairs from observed/simulated interaction outputs

def interaction_dirs_for_pair_window(id_pair: Tuple[int, int], window_h: int) -> List[Tuple[int, Path]]:
    cfg = RUNS[window_h]
    pdir = pair_folder(cfg["run_dir"], id_pair)
    batch = resolve_batch_label_for_pair(pdir, cfg["batch_label_candidates"])
    out = []
    for sl, sdir in list_slice_dirs(pdir):
        int_dir = sdir / batch / "interactions"
        if int_dir.exists():
            out.append((sl, int_dir))
    return out

def load_kind_for_pair_window(
    id_pair: Tuple[int, int],
    window_h: int,
    kind: str,
    observed: bool,
    bins: List[str] = BIN_LABELS,
    max_mc: int = MAX_MC_COMPARE,
) -> pd.DataFrame:
    """Load events or pairs for one pair/window. Adds slice_idx, time_bin, MC_round, observed, source_file."""
    rows = []
    id1, id2 = id_pair
    for sl, int_dir in interaction_dirs_for_pair_window(id_pair, window_h):
        for lab in bins:
            if observed:
                pattern = f"{id1}_{id2}_slice*_{kind}_{lab}_observed_time.parquet"
            else:
                pattern = f"{id1}_{id2}_slice*_{kind}_{lab}_MC*.parquet"
            for fp in sorted(int_dir.glob(pattern)):
                info = parse_interaction_filename(fp)
                if info is None:
                    continue
                if info["kind"] != kind or info["time_bin"] != lab or info["observed"] != observed:
                    continue
                if (not observed) and (info["mc"] is None or info["mc"] < 1 or info["mc"] > max_mc):
                    continue
                df = safe_read_parquet(fp)
                if df.empty:
                    continue
                df = df.copy()
                df["slice_idx"] = info["slice_idx"] if info["slice_idx"] is not None else sl
                df["slice"] = df["slice_idx"] + 1
                df["time_bin"] = lab
                df["observed"] = observed
                if observed:
                    df["MC_round"] = np.nan
                else:
                    df["MC_round"] = int(info["mc"])
                df["source_file"] = str(fp)
                rows.append(df)
    if rows:
        out = pd.concat(rows, ignore_index=True)
    else:
        out = pd.DataFrame()
    if "time_bin" in out.columns:
        out["time_bin"] = out["time_bin"].map(standardize_time_bin_label)
    if "MC_round" in out.columns:
        out["MC_round"] = pd.to_numeric(out["MC_round"], errors="coerce")
    return out

def load_all_results_for_pair_window(id_pair: Tuple[int, int], window_h: int) -> Dict[str, pd.DataFrame]:
    """Load only what is needed for the revised sensitivity analysis.

    - Events: observed + simulation/null model, because these support observed-vs-null
      duration and event-frequency comparisons.
    - Pairs: observed only, because R and M are interpreted for observed interactions
      in the paper. Simulated R/M are intentionally not computed.
    """
    return {
        "obs_events": load_kind_for_pair_window(id_pair, window_h, "events", observed=True),
        "sim_events": load_kind_for_pair_window(id_pair, window_h, "events", observed=False),
        "obs_pairs": load_kind_for_pair_window(id_pair, window_h, "pairs", observed=True),
    }

# Small manifest to confirm files are found before summarizing.
manifest_rows = []
for id_pair in PAIR_IDS:
    for wh in RUNS:
        cfg = RUNS[wh]
        pdir = pair_folder(cfg["run_dir"], id_pair)
        batch = resolve_batch_label_for_pair(pdir, cfg["batch_label_candidates"])
        n_dirs = len(interaction_dirs_for_pair_window(id_pair, wh))
        manifest_rows.append({"pair": pair_id_string(id_pair), "pair_label": pair_label(id_pair),
                              "window_h": wh, "pair_dir": str(pdir), "batch_label_used": batch,
                              "n_interaction_dirs": n_dirs})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(TABLE_DIR / "interaction_folder_manifest.csv", index=False)
display(manifest)

In [7]:
# Event de-duplication and duration/count summaries

def make_hash_key(df: pd.DataFrame, preferred_cols: List[str], include_mc: bool = False) -> pd.Series:
    if df is None or df.empty:
        return pd.Series(dtype="object")
    cols = [c for c in preferred_cols if c in df.columns]
    if include_mc and "MC_round" in df.columns and "MC_round" not in cols:
        cols = ["MC_round"] + cols
    if not cols:
        # Fall back to all non-bookkeeping scalar columns.
        bad = {"source_file"}
        cols = [c for c in df.columns if c not in bad and not str(c).lower().startswith("geometry")]
    tmp = df[cols].copy()
    for c in tmp.columns:
        if pd.api.types.is_datetime64_any_dtype(tmp[c]):
            tmp[c] = pd.to_datetime(tmp[c], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
        else:
            tmp[c] = tmp[c].astype(str)
    return pd.util.hash_pandas_object(tmp, index=False).astype(str)

EVENT_KEY_COLS = [
    "p1", "p1_start", "p1_end", "p1_t_start", "p1_t_end",
    "p2", "p2_start", "p2_end", "p2_t_start", "p2_t_end",
    "difference", "time_bin",
]

def deduplicate_events(df: pd.DataFrame, include_mc: bool) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df.copy()
    out = df.copy()
    out["__event_key"] = make_hash_key(out, EVENT_KEY_COLS, include_mc=include_mc)
    subset = ["__event_key"]
    if include_mc and "MC_round" in out.columns:
        subset = ["MC_round", "__event_key"]
    return out.drop_duplicates(subset=subset).copy()

def interval_union_hours_for_group(df: pd.DataFrame, start, end) -> float:
    if df is None or df.empty:
        return 0.0
    cols = find_interval_cols(df)
    if cols is None:
        c = duration_column(df)
        if c is None:
            return 0.0
        return float(duration_col_to_hours(df[c], c).sum())
    s = to_datetime_series(df[cols[0]])
    e = to_datetime_series(df[cols[1]])
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    intervals = []
    for a, b in zip(s, e):
        if pd.isna(a) or pd.isna(b):
            continue
        a = max(a, start)
        b = min(b, end)
        if b > a:
            intervals.append((a, b))
    if not intervals:
        return 0.0
    intervals.sort(key=lambda x: x[0])
    merged = []
    cur_s, cur_e = intervals[0]
    for a, b in intervals[1:]:
        if a <= cur_e:
            cur_e = max(cur_e, b)
        else:
            merged.append((cur_s, cur_e))
            cur_s, cur_e = a, b
    merged.append((cur_s, cur_e))
    return sum((b - a).total_seconds() for a, b in merged) / 3600.0

def sum_duration_hours(df: pd.DataFrame, start, end, use_interval_union: bool = False) -> float:
    if df is None or df.empty:
        return 0.0
    if use_interval_union:
        return interval_union_hours_for_group(df, start, end)
    c = duration_column(df)
    if c is not None:
        return float(duration_col_to_hours(df[c], c).sum())
    # If no duration column but interval columns exist, fall back to interval duration without union.
    cols = find_interval_cols(df)
    if cols is not None:
        s = to_datetime_series(df[cols[0]]).clip(lower=pd.to_datetime(start), upper=pd.to_datetime(end))
        e = to_datetime_series(df[cols[1]]).clip(lower=pd.to_datetime(start), upper=pd.to_datetime(end))
        dur = (e - s).dt.total_seconds().clip(lower=0) / 3600.0
        return float(dur.fillna(0).sum())
    return 0.0

def summarize_event_metrics_for_pair_window(
    id_pair: Tuple[int, int],
    window_h: int,
    obs_events: pd.DataFrame,
    sim_events: pd.DataFrame,
    common_start,
    common_end,
) -> pd.DataFrame:
    rows = []

    # For event frequency, crop by reference time so all windows are summarized over
    # the same common temporal coverage.
    obs_events_ref = deduplicate_events(
        crop_by_common_coverage(obs_events, common_start, common_end, use_interval_overlap=False),
        include_mc=False,
    )
    sim_events_ref = deduplicate_events(
        crop_by_common_coverage(sim_events, common_start, common_end, use_interval_overlap=False),
        include_mc=True,
    )

    # For concurrent duration, also keep events overlapping the boundary and clip
    # intervals to the common coverage before computing interval-union duration.
    obs_events_interval = deduplicate_events(
        crop_by_common_coverage(obs_events, common_start, common_end, use_interval_overlap=True),
        include_mc=False,
    )
    sim_events_interval = deduplicate_events(
        crop_by_common_coverage(sim_events, common_start, common_end, use_interval_overlap=True),
        include_mc=True,
    )

    for lab in BIN_LABELS:
        obs_bin_count = obs_events_ref[obs_events_ref.get("time_bin", pd.Series(dtype="object")).eq(lab)].copy() if not obs_events_ref.empty else pd.DataFrame()
        sim_bin_count = sim_events_ref[sim_events_ref.get("time_bin", pd.Series(dtype="object")).eq(lab)].copy() if not sim_events_ref.empty else pd.DataFrame()

        if lab == CONCURRENT_BIN:
            obs_bin_dur = obs_events_interval[obs_events_interval.get("time_bin", pd.Series(dtype="object")).eq(lab)].copy() if not obs_events_interval.empty else pd.DataFrame()
            sim_bin_dur = sim_events_interval[sim_events_interval.get("time_bin", pd.Series(dtype="object")).eq(lab)].copy() if not sim_events_interval.empty else pd.DataFrame()
        else:
            obs_bin_dur = obs_bin_count
            sim_bin_dur = sim_bin_count

        # Observed count and duration.
        obs_count = int(len(obs_bin_count))
        obs_duration_h = sum_duration_hours(
            obs_bin_dur,
            common_start,
            common_end,
            use_interval_union=(lab == CONCURRENT_BIN),
        )

        # Simulated event count by MC, with missing MCs treated as zero.
        if not sim_bin_count.empty and "MC_round" in sim_bin_count.columns:
            sim_counts = sim_bin_count.groupby("MC_round").size().reindex(MC_INDEX, fill_value=0).astype(float)
        else:
            sim_counts = pd.Series(0.0, index=MC_INDEX)

        # Simulated duration by MC.
        if not sim_bin_dur.empty and "MC_round" in sim_bin_dur.columns:
            dur_by_mc = []
            for mc in MC_INDEX:
                g = sim_bin_dur[sim_bin_dur["MC_round"].eq(mc)]
                dur_by_mc.append(sum_duration_hours(g, common_start, common_end, use_interval_union=(lab == CONCURRENT_BIN)))
            sim_durations = pd.Series(dur_by_mc, index=MC_INDEX, dtype=float)
        else:
            sim_durations = pd.Series(0.0, index=MC_INDEX)

        rows.append({
            "pair": pair_id_string(id_pair),
            "pair_label": pair_label(id_pair),
            "window_h": int(window_h),
            "time_bin": lab,
            "common_start": pd.to_datetime(common_start),
            "common_end": pd.to_datetime(common_end),
            "observed_events": obs_count,
            "observed_duration_h": obs_duration_h,
            "sim_events_mean": float(sim_counts.mean()),
            "sim_events_median": float(sim_counts.median()),
            "sim_events_q025": float(sim_counts.quantile(0.025)),
            "sim_events_q975": float(sim_counts.quantile(0.975)),
            "sim_duration_h_mean": float(sim_durations.mean()),
            "sim_duration_h_median": float(sim_durations.median()),
            "sim_duration_h_q025": float(sim_durations.quantile(0.025)),
            "sim_duration_h_q975": float(sim_durations.quantile(0.975)),
            "n_obs_event_rows_after_crop_dedup": int(len(obs_bin_count)),
            "n_sim_event_rows_after_crop_dedup": int(len(sim_bin_count)),
        })
    return pd.DataFrame(rows)

print("Event summary helpers ready.")

Event summary helpers ready.


In [8]:
# Pair-file helpers for paired PPA counts and R/M metrics

PPA_SIDE1_CANDIDATE_SETS = [
    ["p1", "p1_t_start", "p1_t_end"],
    ["p1", "p1_start", "p1_end"],
    ["p1_id", "p1_t_start", "p1_t_end"],
    ["id1", "p1_t_start", "p1_t_end"],
]
PPA_SIDE2_CANDIDATE_SETS = [
    ["p2", "p2_t_start", "p2_t_end"],
    ["p2", "p2_start", "p2_end"],
    ["p2_id", "p2_t_start", "p2_t_end"],
    ["id2", "p2_t_start", "p2_t_end"],
]

def find_ppa_key_columns(df: pd.DataFrame) -> Tuple[List[str], List[str], str]:
    for s1, s2 in zip(PPA_SIDE1_CANDIDATE_SETS, PPA_SIDE2_CANDIDATE_SETS):
        if all(c in df.columns for c in s1) and all(c in df.columns for c in s2):
            return s1, s2, "canonical"
    # Fallback: use any p1*/p2* columns that include time/id-like fields.
    p1_cols = [c for c in df.columns if str(c).startswith("p1") and any(k in str(c).lower() for k in ["p1", "start", "end", "time", "id"])]
    p2_cols = [c for c in df.columns if str(c).startswith("p2") and any(k in str(c).lower() for k in ["p2", "start", "end", "time", "id"])]
    if p1_cols and p2_cols:
        return p1_cols, p2_cols, "fallback_p_prefix"
    return [], [], "failed"

def make_ppa_side_key(df: pd.DataFrame, cols: List[str], side_prefix: str) -> pd.Series:
    if df.empty or not cols:
        return pd.Series(dtype="object")
    tmp = df[cols].copy()
    for c in tmp.columns:
        if "time" in str(c).lower() or "start" in str(c).lower() or "end" in str(c).lower():
            parsed = pd.to_datetime(tmp[c], errors="coerce")
            if parsed.notna().any():
                tmp[c] = parsed.dt.strftime("%Y-%m-%d %H:%M:%S")
            else:
                tmp[c] = tmp[c].astype(str)
        else:
            tmp[c] = tmp[c].astype(str)
    key = tmp.astype(str).agg("|".join, axis=1)
    return side_prefix + ":" + key

def paired_ppa_count(df: pd.DataFrame) -> Tuple[int, int, str]:
    """Return paired_ppa, pair_rows, method."""
    if df is None or df.empty:
        return 0, 0, "no_pair_rows"
    d = df.copy()
    s1, s2, method = find_ppa_key_columns(d)
    if method == "failed":
        return np.nan, int(len(d)), "failed_infer_ppa_columns"
    key1 = make_ppa_side_key(d, s1, "side1")
    key2 = make_ppa_side_key(d, s2, "side2")
    paired = pd.concat([key1, key2], ignore_index=True).dropna().nunique()
    pair_rows = d.drop_duplicates(subset=[c for c in (s1 + s2) if c in d.columns]).shape[0]
    return int(paired), int(pair_rows), method

def compute_pair_level_opportunity(id_pair: Tuple[int, int], common_start, common_end) -> dict:
    """Compute all_ppa and common-coverage MCP overlap area for one pair."""
    sub = tracking_df[tracking_df[ID_COL].isin(list(id_pair))].copy()
    sub = sub[(sub["__time"] >= pd.to_datetime(common_start)) & (sub["__time"] < pd.to_datetime(common_end))].copy()
    counts = sub.groupby(ID_COL).size().to_dict()
    all_ppa = int(max(sum(counts.get(i, 0) for i in id_pair) - 2, 0))

    xcol = first_existing_column(sub, X_COL_CANDIDATES)
    ycol = first_existing_column(sub, Y_COL_CANDIDATES)
    overlap_km2 = np.nan
    area1_km2 = np.nan
    area2_km2 = np.nan
    if xcol is not None and ycol is not None and len(sub) >= 6:
        try:
            # If projected columns are available, use them directly. Otherwise use geopandas to project lon/lat.
            if xcol.startswith("proj") or ycol.startswith("proj") or xcol.lower() in ["x"] or ycol.lower() in ["y"]:
                from shapely.geometry import MultiPoint
                polys = {}
                for indiv in id_pair:
                    pts = sub[sub[ID_COL].eq(indiv)][[xcol, ycol]].dropna().values
                    if len(pts) >= 3:
                        polys[indiv] = MultiPoint([(float(x), float(y)) for x, y in pts]).convex_hull
                if len(polys) == 2:
                    area1_km2 = polys[id_pair[0]].area / 1e6
                    area2_km2 = polys[id_pair[1]].area / 1e6
                    overlap_km2 = polys[id_pair[0]].intersection(polys[id_pair[1]]).area / 1e6
            elif HAS_GEOPANDAS:
                gdf = gpd.GeoDataFrame(sub, geometry=gpd.points_from_xy(sub[xcol], sub[ycol]), crs="EPSG:4326").to_crs(epsg=UTM_EPSG)
                polys = {}
                for indiv in id_pair:
                    pts = gdf[gdf[ID_COL].eq(indiv)]
                    if len(pts) >= 3:
                        polys[indiv] = pts.unary_union.convex_hull
                if len(polys) == 2:
                    area1_km2 = polys[id_pair[0]].area / 1e6
                    area2_km2 = polys[id_pair[1]].area / 1e6
                    overlap_km2 = polys[id_pair[0]].intersection(polys[id_pair[1]]).area / 1e6
        except Exception as e:
            print(f"[warning] overlap area failed for {pair_label(id_pair)}: {e}")
    return {
        "all_ppa": all_ppa,
        "overlapped_area_km2": overlap_km2,
        "area1_km2": area1_km2,
        "area2_km2": area2_km2,
        "n_points_total": int(sum(counts.get(i, 0) for i in id_pair)),
        f"n_points_{id_pair[0]}": int(counts.get(id_pair[0], 0)),
        f"n_points_{id_pair[1]}": int(counts.get(id_pair[1], 0)),
    }

def summarize_pair_files_for_pair_window(
    id_pair: Tuple[int, int],
    window_h: int,
    obs_pairs: pd.DataFrame,
    common_start,
    common_end,
) -> pd.DataFrame:
    """Return observed paired-PPA rows for each time bin.

    R and M are observed-only in this paper. Simulation files are not used here;
    simulation outputs are only summarized as null distributions for event frequency
    and duration.
    """
    obs_pairs = crop_by_common_coverage(obs_pairs, common_start, common_end, use_interval_overlap=False)
    opp = compute_pair_level_opportunity(id_pair, common_start, common_end)

    obs_rows = []
    for lab in BIN_LABELS:
        obs_bin = obs_pairs[obs_pairs.get("time_bin", pd.Series(dtype="object")).eq(lab)].copy() if not obs_pairs.empty else pd.DataFrame()
        paired, pair_rows, method = paired_ppa_count(obs_bin)
        obs_rows.append({
            "pair": pair_id_string(id_pair), "pair_label": pair_label(id_pair),
            "window_h": int(window_h), "time_bin": lab,
            "observed_paired_ppa": paired, "observed_pair_rows": pair_rows,
            "observed_ppa_count_method": method,
            **opp,
        })
    return pd.DataFrame(obs_rows)

print("Pair/PPA and opportunity helpers ready.")

Pair/PPA and opportunity helpers ready.


In [9]:
# Compute observed-only R and M from paired-PPA summaries

def add_R_M_observed(obs_pairs_summary: pd.DataFrame) -> pd.DataFrame:
    out = obs_pairs_summary.copy()
    out["observed_paired_ppa"] = pd.to_numeric(out["observed_paired_ppa"], errors="coerce")
    out["all_ppa"] = pd.to_numeric(out["all_ppa"], errors="coerce")
    out["overlapped_area_km2"] = pd.to_numeric(out["overlapped_area_km2"], errors="coerce")
    out["observed_ppa_prop"] = np.where(out["all_ppa"] > 0, out["observed_paired_ppa"] / out["all_ppa"], np.nan)
    out["observed_D"] = np.where(out["overlapped_area_km2"] > 0, out["observed_ppa_prop"] / out["overlapped_area_km2"], np.nan)
    out["observed_D0"] = out.groupby(["pair", "window_h"], dropna=False)["observed_D"].transform("mean")
    denom = out["observed_D"] + out["observed_D0"]
    out["observed_R"] = np.where(denom > 1e-12, (out["observed_D"] - out["observed_D0"]) / (denom + 1e-12), np.nan)
    out["observed_R"] = pd.to_numeric(out["observed_R"], errors="coerce").clip(-1, 1)

    # M / PC1_norm within one pair × window across time bins.
    out["observed_ln_ppa"] = np.log(out["observed_ppa_prop"].fillna(0) + 1.0)
    out["observed_ln_area"] = np.log(out["overlapped_area_km2"].fillna(0) + 1.0)
    pcs = []
    for _, g in out.groupby(["pair", "window_h"], dropna=False):
        gg = g.copy()
        z_n = (gg["observed_ln_ppa"] - gg["observed_ln_ppa"].mean()) / (gg["observed_ln_ppa"].std(ddof=0) + 1e-12)
        z_a = (gg["observed_ln_area"] - gg["observed_ln_area"].mean()) / (gg["observed_ln_area"].std(ddof=0) + 1e-12)
        pc1 = z_n + z_a
        mn, mx = pc1.min(), pc1.max()
        if pd.notna(mn) and pd.notna(mx) and mx > mn:
            gg["observed_M"] = 2 * (pc1 - mn) / (mx - mn) - 1
        else:
            gg["observed_M"] = 0.0
        pcs.append(gg)
    return pd.concat(pcs, ignore_index=True) if pcs else out

print("Observed-only R/M helpers ready.")

Observed-only R/M helpers ready.


In [ ]:
# Run all pair × window summaries

all_event_summaries = []
all_obs_pair_summaries = []

for id_pair in PAIR_IDS:
    plab = pair_label(id_pair)
    cov_row = common_coverage[common_coverage["pair_label"].eq(plab)].iloc[0]
    common_start = pd.to_datetime(cov_row["common_start"])
    common_end = pd.to_datetime(cov_row["common_end"])
    print(f"\n=== {plab} ({pair_id_string(id_pair)}) | common coverage {common_start} to {common_end} ===")

    for wh in sorted(RUNS):
        print(f"Loading {plab}, {wh} h ...")
        res = load_all_results_for_pair_window(id_pair, wh)

        ev_sum = summarize_event_metrics_for_pair_window(
            id_pair=id_pair,
            window_h=wh,
            obs_events=res["obs_events"],
            sim_events=res["sim_events"],
            common_start=common_start,
            common_end=common_end,
        )
        all_event_summaries.append(ev_sum)

        obs_pair = summarize_pair_files_for_pair_window(
            id_pair=id_pair,
            window_h=wh,
            obs_pairs=res["obs_pairs"],
            common_start=common_start,
            common_end=common_end,
        )
        obs_pair_rm = add_R_M_observed(obs_pair)
        all_obs_pair_summaries.append(obs_pair_rm)

        print("  obs_events rows:", len(res["obs_events"]), "sim_events rows:", len(res["sim_events"]))
        print("  obs_pairs rows :", len(res["obs_pairs"]))

# Combine.
event_summary = pd.concat(all_event_summaries, ignore_index=True)
obs_pair_rm_summary = pd.concat(all_obs_pair_summaries, ignore_index=True)

# Merge event summaries with observed-only R/M summaries.
metrics_summary = event_summary.merge(
    obs_pair_rm_summary[[
        "pair", "pair_label", "window_h", "time_bin",
        "observed_paired_ppa", "observed_pair_rows", "all_ppa", "overlapped_area_km2",
        "observed_ppa_prop", "observed_D", "observed_D0", "observed_R", "observed_M",
        "observed_ppa_count_method", "n_points_total",
    ]],
    on=["pair", "pair_label", "window_h", "time_bin"],
    how="left",
)

# Order rows.
metrics_summary["time_bin"] = pd.Categorical(metrics_summary["time_bin"], categories=BIN_LABELS, ordered=True)
metrics_summary = metrics_summary.sort_values(["pair_label", "window_h", "time_bin"]).reset_index(drop=True)
metrics_summary["time_bin"] = metrics_summary["time_bin"].astype(str)

# Save outputs.
metrics_summary.to_csv(TABLE_DIR / "sensitivity_pair_window_bin_observed_RM_sim_medians.csv", index=False)
obs_pair_rm_summary.to_csv(TABLE_DIR / "observed_pair_window_bin_RM_components.csv", index=False)

print("Saved main metrics table:", TABLE_DIR / "sensitivity_pair_window_bin_observed_RM_sim_medians.csv")
display(metrics_summary.head(30))

In [ ]:
# Consistency tables: differences from 1000 h

def add_difference_from_1000(df: pd.DataFrame, value_col: str, out_col: str) -> pd.DataFrame:
    base = df[df["window_h"].eq(1000)][["pair", "pair_label", "time_bin", value_col]].rename(columns={value_col: f"{value_col}_1000"})
    out = df.merge(base, on=["pair", "pair_label", "time_bin"], how="left")
    out[out_col] = pd.to_numeric(out[value_col], errors="coerce") - pd.to_numeric(out[f"{value_col}_1000"], errors="coerce")
    return out.drop(columns=[f"{value_col}_1000"])

consistency = metrics_summary.copy()
for value_col, out_col in [
    ("observed_events", "observed_events_minus_1000"),
    ("sim_events_median", "sim_events_median_minus_1000"),
    ("observed_duration_h", "observed_duration_h_minus_1000"),
    ("sim_duration_h_median", "sim_duration_h_median_minus_1000"),
    ("observed_R", "observed_R_minus_1000"),
    ("observed_M", "observed_M_minus_1000"),
]:
    if value_col in consistency.columns:
        consistency = add_difference_from_1000(consistency, value_col, out_col)

consistency.to_csv(TABLE_DIR / "sensitivity_consistency_differences_from_1000.csv", index=False)

print("Event frequency median differences from 1000 h:")
display(consistency[["pair_label", "time_bin", "window_h", "observed_events", "sim_events_median", "sim_events_median_minus_1000"]].head(80))

print("Duration median differences from 1000 h:")
display(consistency[["pair_label", "time_bin", "window_h", "observed_duration_h", "sim_duration_h_median", "sim_duration_h_median_minus_1000"]].head(80))

print("Saved:", TABLE_DIR / "sensitivity_consistency_differences_from_1000.csv")

In [12]:
# Figure style helpers

WINDOW_ORDER = [500, 1000, 2000]
WINDOW_LABELS = {500: "500 h", 1000: "1000 h", 2000: "2000 h"}

# Neutral grayscale style to avoid confusion with main-text state colors.
WINDOW_STYLES = {
    500:  {"color": "#7A7A7A", "marker": "o", "linestyle": "-"},
    1000: {"color": "#222222", "marker": "s", "linestyle": "-"},
    2000: {"color": "#B0B0B0", "marker": "^", "linestyle": "-"},
}

OBS_STYLE = {"linestyle": "-", "linewidth": 1.9, "alpha": 1.0}
SIM_STYLE = {"linestyle": "--", "linewidth": 1.7, "alpha": 0.95}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def format_axis(ax):
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="x", rotation=35)

METRIC_SPECS = {
    "event_frequency": {
        "obs": "observed_events",
        "sim": "sim_events_median",
        "ylabel": "Event frequency",
        "title": "Event frequency",
        "filename": "event_frequency",
    },
    "duration": {
        "obs": "observed_duration_h",
        "sim": "sim_duration_h_median",
        "ylabel": "Total duration (h)",
        "title": "Duration",
        "filename": "duration_h",
    },
    "R": {
        "obs": "observed_R",
        "sim": None,
        "ylabel": "R",
        "title": "Observed R metric",
        "filename": "observed_R_metric",
    },
    "M": {
        "obs": "observed_M",
        "sim": None,
        "ylabel": "M",
        "title": "Observed M metric",
        "filename": "observed_M_metric",
    },
}

print("Figure style ready.")

Figure style ready.


In [20]:
# ============================================================
# Sensitivity figures:
# x-axis = temporal partition length; lines = pairs
# Add error bars for simulated median panel
# ============================================================

PAIR_STYLES = {
    "FT1-ML2": {"color": "#000000", "marker": "o", "linestyle": "-"},
    "FT1-ML4": {"color": "#555555", "marker": "s", "linestyle": "--"},
    "MT3-ML2": {"color": "#999999", "marker": "^", "linestyle": "-."},
}

SIM_ERROR_STYLE = {
    "elinewidth": 1.0,
    "capsize": 3,
    "capthick": 0.9,
    "alpha": 0.65,
}

SIM_ERROR_COLS = {
    "event_frequency": ("sim_events_q025", "sim_events_q975"),
    "duration": ("sim_duration_h_q025", "sim_duration_h_q975"),
}

def plot_metric_across_windows(
    summary_df: pd.DataFrame,
    metric_key: str,
    time_bin: str = "0-1h",
    save_dir: Path = FIG_DIR,
    show_simulated: bool = True,
    add_sim_errorbar: bool = False,
):
    """
    Plot one metric across temporal partition lengths.

    x-axis: 500 h, 1000 h, 2000 h
    lines: tiger-leopard pairs

    Observed panel:
        line only.

    Simulated panel:
        median line
    """

    spec = METRIC_SPECS[metric_key]
    obs_col = spec["obs"]
    sim_col = spec.get("sim", None)

    d = summary_df.copy()

    if "time_bin" in d.columns and time_bin is not None:
        d = d[d["time_bin"].eq(time_bin)].copy()

    d["window_h"] = pd.Categorical(
        d["window_h"],
        categories=WINDOW_ORDER,
        ordered=True
    )

    has_sim = (
        show_simulated
        and sim_col is not None
        and sim_col in d.columns
    )

    if has_sim:
        fig, axes = plt.subplots(
            1, 2,
            figsize=(7.2, 3.35),
            sharex=True,
            sharey=True
        )
        panel_specs = [
            ("Observed", obs_col, axes[0], False),
            ("Null-model median", sim_col, axes[1], True),
        ]
    else:
        fig, ax = plt.subplots(figsize=(3.9, 3.35))
        axes = [ax]
        panel_specs = [("Observed", obs_col, ax, False)]

    for panel_title, y_col, ax, is_sim_panel in panel_specs:
        for pair_lab in [pair_label(p) for p in PAIR_IDS]:
            g = (
                d[d["pair_label"].eq(pair_lab)]
                .sort_values("window_h")
                .copy()
            )

            g = (
                g.set_index("window_h")
                 .reindex(WINDOW_ORDER)
                 .reset_index()
            )

            style = PAIR_STYLES.get(
                pair_lab,
                {"color": "#555555", "marker": "o", "linestyle": "-"}
            )

            x_vals = np.arange(len(WINDOW_ORDER))
            x_labels = [WINDOW_LABELS[w] for w in WINDOW_ORDER]
            y_vals = pd.to_numeric(g[y_col], errors="coerce")

            ax.plot(
                x_vals,
                y_vals,
                label=pair_lab,
                color=style["color"],
                marker=style["marker"],
                linestyle=style["linestyle"],
                linewidth=1.8,
                markersize=5.5,
                markerfacecolor="white",
                markeredgewidth=1.1,
            )

            # Add error bars only for simulated median panel.
            if is_sim_panel and add_sim_errorbar and metric_key in SIM_ERROR_COLS:
                low_col, high_col = SIM_ERROR_COLS[metric_key]

                if low_col in g.columns and high_col in g.columns:
                    y_low = pd.to_numeric(g[low_col], errors="coerce")
                    y_high = pd.to_numeric(g[high_col], errors="coerce")

                    lower_err = y_vals - y_low
                    upper_err = y_high - y_vals

                    # Avoid negative error lengths if there are tiny numerical issues.
                    lower_err = lower_err.clip(lower=0)
                    upper_err = upper_err.clip(lower=0)

                    ax.errorbar(
                        x_vals,
                        y_vals,
                        yerr=np.vstack([lower_err, upper_err]),
                        fmt="none",
                        color=style["color"],
                        elinewidth=SIM_ERROR_STYLE["elinewidth"],
                        capsize=SIM_ERROR_STYLE["capsize"],
                        capthick=SIM_ERROR_STYLE["capthick"],
                        alpha=SIM_ERROR_STYLE["alpha"],
                    )

        ax.set_title(panel_title)
        ax.set_xlabel("Temporal partition length")
        ax.set_ylabel(spec["ylabel"])
        ax.set_xticks(np.arange(len(WINDOW_ORDER)))
        ax.set_xticklabels([WINDOW_LABELS[w] for w in WINDOW_ORDER])
        format_axis(ax)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        frameon=False,
        ncol=3,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.04)
    )

    if time_bin is not None:
        title_bin = BIN_DISPLAY.get(time_bin, time_bin)
        fig.suptitle(
            f"{spec['title']} ({title_bin})",
            y=1.03,
            fontsize=11
        )
        safe_bin = str(time_bin).replace("-", "_").replace(" ", "").replace("/", "_")
    else:
        fig.suptitle(
            f"{spec['title']}",
            y=1.03,
            fontsize=11
        )
        safe_bin = "all_bins"

    fig.tight_layout(rect=[0, 0.08, 1, 1])

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    out_png = save_dir / f"sensitivity_{spec['filename']}_{safe_bin}_across_windows.png"

    fig.savefig(out_png, dpi=500, bbox_inches="tight")
    fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")

    plt.show()
    return out_png

In [ ]:
fig_path = plot_metric_across_windows(
    metrics_summary,
    metric_key="duration",
    time_bin="0-1h",
    save_dir=FIG_DIR / "paper_duration",
    show_simulated=True,
    add_sim_errorbar=False,
)

print(fig_path)